<a href="https://colab.research.google.com/github/Tolverne/HOM_simulation/blob/main/HOM_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install perceval-quandela



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.8/432.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.1 MB/s eta 0:00:00


In [46]:
import perceval as pcvl
from perceval.algorithm import Sampler
import argparse
import csv
import math
from pathlib import Path
from typing import Dict, List

import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os

# Define the output directory path
outdir = Path('/content/drive/MyDrive/HOM_simulation')

# Create the folder if it doesn't exist
if not os.path.exists(outdir):
    os.makedirs(outdir)

Mounted at /content/drive


In [19]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run a finite-shot HOM delay sweep in Perceval.")
    parser.add_argument("--sigma-t", type=float, default=1.0, help="RMS temporal width of each photon wavepacket.")
    parser.add_argument("--tau-min", type=float, default=-4.0, help="Minimum delay.")
    parser.add_argument("--tau-max", type=float, default=4.0, help="Maximum delay.")
    parser.add_argument("--npts", type=int, default=81, help="Number of delay points.")
    parser.add_argument("--nsamples", type=int, default=4000, help="Number of samples per delay point.")
    parser.add_argument("--backend", type=str, default="SLOS", help="Perceval backend name.")
    parser.add_argument("--outdir", type=str, default="outputs_hom_sampled", help="Directory for CSV and figures.")
    return parser.parse_args([])

In [23]:
def gaussian_eta(tau: float, sigma_t: float) -> float:
    """Squared overlap eta = |<psi1|psi2>|^2 for Gaussian temporal modes."""
    return math.exp(-(tau ** 2) / (2.0 * sigma_t ** 2))

In [25]:
def gaussian_indistinguishability(tau: float, sigma_t: float) -> float:
    """
    Field overlap magnitude for two Gaussian temporal modes.

    If eta = |<psi1|psi2>|^2 = exp[-tau^2/(2 sigma_t^2)],
    then the amplitude overlap used in the Perceval StateVector is

        indist = sqrt(eta) = exp[-tau^2/(4 sigma_t^2)]
    """
    return math.exp(-(tau ** 2) / (4.0 * sigma_t ** 2))

In [26]:
def canonical_counts(counts: Dict) -> Dict[str, int]:
    """Convert a Perceval count dictionary into a plain {state_string: count} dict."""
    return {str(k): int(v) for k, v in counts.items()}


In [27]:
def canonical_dist(dist: Dict) -> Dict[str, float]:
    """Convert a Perceval distribution into a plain {state_string: probability} dict."""
    out = {str(k): float(v) for k, v in dist.items()}
    total = sum(out.values())
    if total <= 0:
        return out
    return {k: v / total for k, v in out.items()}

In [24]:
def wilson_interval_half_width(k: int, n: int, z: float = 1.96) -> float:
    """95% Wilson interval half-width for a binomial proportion."""
    if n <= 0:
        return float("nan")
    phat = k / n
    denom = 1.0 + z ** 2 / n
    centre = (phat + z ** 2 / (2.0 * n)) / denom
    radius = z * math.sqrt((phat * (1.0 - phat) + z ** 2 / (4.0 * n)) / n) / denom
    # half-width relative to centre; symmetric plotting uses radius directly
    return radius

In [28]:
def compute_indistinguishability(annotation_1, annotation_2, sigma_t: float) -> float:
    """
    Compute the field overlap from two annotations containing a time tag `t`.
    """
    t1 = float(np.real(annotation_1["t"]))
    t2 = float(np.real(annotation_2["t"]))
    tau = t2 - t1
    return gaussian_indistinguishability(tau, sigma_t)

In [29]:
def convert_annotated_state(state, sigma_t: float):
    """
    Convert a 2-photon AnnotatedFockState into a simulable StateVector.

    Structure follows the user's existing script: clear annotations ->
    indistinguishable FockState, plus a NoisyFockState component tagged [0, 1].
    """
    if state.n != 2:
        raise ValueError("This converter is written for the 2-photon HOM case only.")

    photon_1 = state.get_photon_annotation(0)
    photon_2 = state.get_photon_annotation(1)

    indist = compute_indistinguishability(photon_1, photon_2, sigma_t)
    noise_amp = math.sqrt(max(0.0, 1.0 - indist ** 2))

    indist_state = state.clear_annotations()
    noisy_state = pcvl.BasicState(list(state), [0, 1])

    return indist_state * indist + noisy_state * noise_amp

In [30]:
class HOMSimulatorSampled:
    def __init__(self, backend: str = "SLOS") -> None:
        self.backend = backend
        self.circuit = pcvl.BS()

    def _build_processor(self, converted_input):
        processor = pcvl.Processor(self.backend, self.circuit)
        processor.min_detected_photons_filter(2)
        processor.with_input(converted_input)
        return processor

    def simulate_delay(self, tau: float, sigma_t: float, nsamples: int) -> Dict[str, float]:
        annotated_input = pcvl.BasicState(f"|{{t:0}},{{t:{tau}}}>")
        converted_input = convert_annotated_state(annotated_input, sigma_t=sigma_t)

        processor = self._build_processor(converted_input)
        sampler = Sampler(processor)

        counts = canonical_counts(sampler.sample_count(nsamples)["results"])
        detected_total = int(sum(counts.values()))
        if detected_total <= 0:
            raise RuntimeError("No samples were returned by sample_count().")

        c11 = counts.get("|1,1>", 0)
        c20 = counts.get("|2,0>", 0)
        c02 = counts.get("|0,2>", 0)

        p11 = c11 / detected_total
        p20 = c20 / detected_total
        p02 = c02 / detected_total

        eta = gaussian_eta(tau, sigma_t)
        indist = math.sqrt(eta)

        visibility = (0.5 - p11) / 0.5

        analytic_p11 = 0.5 * (1.0 - eta)
        analytic_p20 = 0.25 * (1.0 + eta)
        analytic_p02 = 0.25 * (1.0 + eta)
        analytic_visibility = eta

        p11_err = wilson_interval_half_width(c11, detected_total)
        p20_err = wilson_interval_half_width(c20, detected_total)
        p02_err = wilson_interval_half_width(c02, detected_total)
        visibility_err = 2.0 * p11_err

        return {
            "tau": float(tau),
            "sigma_t": float(sigma_t),
            "nsamples_requested": int(nsamples),
            "nsamples_recorded": int(detected_total),
            "eta": float(eta),
            "indist_amplitude": float(indist),
            "annotated_input": str(annotated_input),
            "converted_input": str(converted_input),
            "C11": int(c11),
            "C20": int(c20),
            "C02": int(c02),
            "P11_sim": float(p11),
            "P20_sim": float(p20),
            "P02_sim": float(p02),
            "P11_err": float(p11_err),
            "P20_err": float(p20_err),
            "P02_err": float(p02_err),
            "visibility_sim": float(visibility),
            "visibility_err": float(visibility_err),
            "P11_analytic": float(analytic_p11),
            "P20_analytic": float(analytic_p20),
            "P02_analytic": float(analytic_p02),
            "visibility_analytic": float(analytic_visibility),
        }


In [33]:
def write_summary(rows: List[Dict[str, float]], summary_path: Path) -> None:
    taus = np.array([r["tau"] for r in rows], dtype=float)
    p11 = np.array([r["P11_sim"] for r in rows], dtype=float)
    vis = np.array([r["visibility_sim"] for r in rows], dtype=float)
    eta = np.array([r["eta"] for r in rows], dtype=float)
    nrec = np.array([r["nsamples_recorded"] for r in rows], dtype=int)

    min_idx = int(np.argmin(p11))
    max_idx = int(np.argmax(p11))

    text = f"""HOM delay sweep summary (finite-shot sampling)
=======================================
Number of points: {len(rows)}
Delay range: [{taus.min():.6g}, {taus.max():.6g}]
Average recorded samples per point: {nrec.mean():.2f}

Coincidence probability P(|1,1>)
--------------------------------
Minimum: {p11[min_idx]:.10f} at tau = {taus[min_idx]:.10f}
Maximum: {p11[max_idx]:.10f} at tau = {taus[max_idx]:.10f}

Visibility
----------
Maximum sampled visibility: {vis.max():.10f}
Minimum sampled visibility: {vis.min():.10f}

Overlap
-------
Maximum eta: {eta.max():.10f}
Minimum eta: {eta.min():.10f}

Notes
-----
These plotted simulation values come from sample_count rather than probs, so
statistical scatter is expected. The analytic curves remain the ideal HOM model.
"""
    summary_path.write_text(text, encoding="utf-8")


In [43]:
def write_csv(rows: List[Dict[str, float]], csv_path: Path) -> None:
    if not rows:
        raise ValueError("No rows to write.")
    fieldnames = list(rows[0].keys())
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


In [34]:
def save_probability_plot(rows: List[Dict[str, float]], outbase: Path) -> None:
    tau = np.array([r["tau"] for r in rows])
    p11_sim = np.array([r["P11_sim"] for r in rows])
    p20_sim = np.array([r["P20_sim"] for r in rows])
    p02_sim = np.array([r["P02_sim"] for r in rows])
    p11_err = np.array([r["P11_err"] for r in rows])
    p20_err = np.array([r["P20_err"] for r in rows])
    p02_err = np.array([r["P02_err"] for r in rows])
    p11_an = np.array([r["P11_analytic"] for r in rows])
    p20_an = np.array([r["P20_analytic"] for r in rows])
    p02_an = np.array([r["P02_analytic"] for r in rows])

    plt.figure(figsize=(8.2, 5.4), dpi=200)
    plt.errorbar(tau, p11_sim, yerr=p11_err, fmt='o', ms=3, capsize=2, linewidth=0.8, label=r"$P_{11}$ sampled")
    plt.plot(tau, p11_an, linestyle="--", linewidth=1.6, label=r"$P_{11}$ analytic")
    plt.errorbar(tau, p20_sim, yerr=p20_err, fmt='s', ms=3, capsize=2, linewidth=0.8, label=r"$P_{20}$ sampled")
    plt.plot(tau, p20_an, linestyle="--", linewidth=1.6, label=r"$P_{20}$ analytic")
    plt.errorbar(tau, p02_sim, yerr=p02_err, fmt='^', ms=3, capsize=2, linewidth=0.8, label=r"$P_{02}$ sampled")
    plt.plot(tau, p02_an, linestyle="--", linewidth=1.6, label=r"$P_{02}$ analytic")
    plt.xlabel(r"Delay $\tau$")
    plt.ylabel("Probability")
    plt.title("Hong-Ou-Mandel output probabilities (finite-shot)")
    plt.grid(True, alpha=0.3)
    plt.legend(frameon=False, ncol=2)
    plt.tight_layout()
    plt.savefig(outbase.with_suffix(".png"), bbox_inches="tight")
    plt.savefig(outbase.with_suffix(".pdf"), bbox_inches="tight")
    plt.close()


In [35]:
def save_visibility_plot(rows: List[Dict[str, float]], outbase: Path) -> None:
    tau = np.array([r["tau"] for r in rows])
    vis_sim = np.array([r["visibility_sim"] for r in rows])
    vis_err = np.array([r["visibility_err"] for r in rows])
    vis_an = np.array([r["visibility_analytic"] for r in rows])

    plt.figure(figsize=(8.2, 5.0), dpi=200)
    plt.errorbar(tau, vis_sim, yerr=vis_err, fmt='o', ms=3, capsize=2, linewidth=0.8, label="Sampled")
    plt.plot(tau, vis_an, linestyle="--", linewidth=1.8, label="Analytic")
    plt.xlabel(r"Delay $\tau$")
    plt.ylabel("Visibility")
    plt.title("Hong-Ou-Mandel visibility (finite-shot)")
    plt.grid(True, alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(outbase.with_suffix(".png"), bbox_inches="tight")
    plt.savefig(outbase.with_suffix(".pdf"), bbox_inches="tight")
    plt.close()

In [36]:
def save_overlap_plot(rows: List[Dict[str, float]], outbase: Path) -> None:
    tau = np.array([r["tau"] for r in rows])
    eta = np.array([r["eta"] for r in rows])
    indist = np.array([r["indist_amplitude"] for r in rows])

    plt.figure(figsize=(8.2, 5.0), dpi=200)
    plt.plot(tau, eta, linewidth=2.0, label=r"$\eta(\tau)=|\langle\psi_1|\psi_2\rangle|^2$")
    plt.plot(tau, indist, linestyle="--", linewidth=1.8, label=r"$|\langle\psi_1|\psi_2\rangle|$")
    plt.xlabel(r"Delay $\tau$")
    plt.ylabel("Overlap")
    plt.title("Temporal overlap model used in the HOM simulation")
    plt.grid(True, alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(outbase.with_suffix(".png"), bbox_inches="tight")
    plt.savefig(outbase.with_suffix(".pdf"), bbox_inches="tight")
    plt.close()

In [50]:
def save_circuit_diagram(circuit, outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)

    svg_path = outdir / "hom_circuit.svg"
    png_path = outdir / "hom_circuit.png"

    saved_any = False

    # Best option: native vector export
    try:
        from perceval.rendering.pdisplay import pdisplay_to_file
        pdisplay_to_file(circuit, str(svg_path))
        print(f"Saved circuit diagram: {svg_path.resolve()}")
        saved_any = True
    except Exception as exc:
        print(f"SVG export failed: {exc}")

    # Fallback: display + matplotlib save, if supported by your environment
    if not saved_any:
        try:
            import matplotlib.pyplot as plt
            pcvl.pdisplay(circuit)
            plt.savefig(png_path, bbox_inches="tight", dpi=200)
            plt.close()
            print(f"Saved circuit diagram: {png_path.resolve()}")
            saved_any = True
        except Exception as exc:
            print(f"PNG fallback export failed: {exc}")

    if not saved_any:
        print("Circuit diagram could be displayed, but not saved in this environment.")

In [44]:
args = parse_args()

In [32]:
taus = np.linspace(args.tau_min, args.tau_max, args.npts)
simulator = HOMSimulatorSampled(backend=args.backend)

In [39]:
rows: List[Dict[str, float]] = []
for i, tau in enumerate(taus, start=1):
    row = simulator.simulate_delay(float(tau), sigma_t=args.sigma_t, nsamples=args.nsamples)
    rows.append(row)
    print(f"[{i:>3}/{len(taus)}] tau={tau: .4f}  P11={row['P11_sim']:.4f}±{row['P11_err']:.4f}  V={row['visibility_sim']:.4f}")


[  1/81] tau=-4.0000  P11=0.5042±0.0155  V=-0.0085
[  2/81] tau=-3.9000  P11=0.4978±0.0155  V=0.0045
[  3/81] tau=-3.8000  P11=0.4963±0.0155  V=0.0075
[  4/81] tau=-3.7000  P11=0.5035±0.0155  V=-0.0070
[  5/81] tau=-3.6000  P11=0.4945±0.0155  V=0.0110
[  6/81] tau=-3.5000  P11=0.5080±0.0155  V=-0.0160
[  7/81] tau=-3.4000  P11=0.5040±0.0155  V=-0.0080
[  8/81] tau=-3.3000  P11=0.5085±0.0155  V=-0.0170
[  9/81] tau=-3.2000  P11=0.4970±0.0155  V=0.0060
[ 10/81] tau=-3.1000  P11=0.4930±0.0155  V=0.0140
[ 11/81] tau=-3.0000  P11=0.4943±0.0155  V=0.0115
[ 12/81] tau=-2.9000  P11=0.4920±0.0155  V=0.0160
[ 13/81] tau=-2.8000  P11=0.5015±0.0155  V=-0.0030
[ 14/81] tau=-2.7000  P11=0.4850±0.0155  V=0.0300
[ 15/81] tau=-2.6000  P11=0.4763±0.0155  V=0.0475
[ 16/81] tau=-2.5000  P11=0.4723±0.0155  V=0.0555
[ 17/81] tau=-2.4000  P11=0.4758±0.0155  V=0.0485
[ 18/81] tau=-2.3000  P11=0.4682±0.0155  V=0.0635
[ 19/81] tau=-2.2000  P11=0.4537±0.0154  V=0.0925
[ 20/81] tau=-2.1000  P11=0.4417±0.0154  V=0

In [51]:
write_csv(rows, outdir / "hom_results_sampled.csv")
write_summary(rows, outdir / "hom_summary_sampled.txt")
save_probability_plot(rows, outdir / "hom_probabilities_sampled")
save_visibility_plot(rows, outdir / "hom_visibility_sampled")
save_overlap_plot(rows, outdir / "hom_overlap")
save_circuit_diagram(simulator.circuit, outdir)

print(f"\nSaved results to: {outdir.resolve()}")
print("Files created:")
for p in sorted(outdir.iterdir()):
    print(f"  - {p.name}")

Saved circuit diagram: /content/drive/MyDrive/HOM_simulation/hom_circuit.svg

Saved results to: /content/drive/MyDrive/HOM_simulation
Files created:
  - hom_circuit.svg
  - hom_overlap.pdf
  - hom_overlap.png
  - hom_probabilities_sampled.pdf
  - hom_probabilities_sampled.png
  - hom_results_sampled.csv
  - hom_summary_sampled.txt
  - hom_visibility_sampled.pdf
  - hom_visibility_sampled.png
